# 01 — Exploratory Data Analysis

## Data Source
This notebook reads and explores the two data files:
1. `census-bureau.columns`: column names (one per line)
2. `census-bureau.data`: the actual dataset

## Goals
Examine the data before any preprocessing or modeling decisions; let observations drive subsequent choices.
1. Load the data correctly with proper column names
2. Understand shape, dtypes, and basic statistics
3. Identify missing / special values
4. Explore the target label distribution
5. Examine numerical vs. categorical features
6. Visualize key distributions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data_raw")
COLUMNS_FILE = DATA_DIR / "census-bureau.columns"
DATA_FILE = DATA_DIR / "census-bureau.data"

In [3]:
# Read column names from the columns file
with open(COLUMNS_FILE, 'r') as f:
    columns = [line.strip() for line in f if line.strip()]

print(f'Number of columns defined: {len(columns)}')
print()
for i, col in enumerate(columns, 1):
    print(f'  {i:2d}. {col}')

Number of columns defined: 42

   1. age
   2. class of worker
   3. detailed industry recode
   4. detailed occupation recode
   5. education
   6. wage per hour
   7. enroll in edu inst last wk
   8. marital stat
   9. major industry code
  10. major occupation code
  11. race
  12. hispanic origin
  13. sex
  14. member of a labor union
  15. reason for unemployment
  16. full or part time employment stat
  17. capital gains
  18. capital losses
  19. dividends from stocks
  20. tax filer stat
  21. region of previous residence
  22. state of previous residence
  23. detailed household and family stat
  24. detailed household summary in household
  25. weight
  26. migration code-change in msa
  27. migration code-change in reg
  28. migration code-move within reg
  29. live in this house 1 year ago
  30. migration prev res in sunbelt
  31. num persons worked for employer
  32. family members under 18
  33. country of birth father
  34. country of birth mother
  35. country of birth

In [4]:
# Load the data: comma delimited, no header row in the file
df = pd.read_csv(DATA_FILE, header=None, names=columns)

In [5]:
print("Shape:", df.shape)
print()
print("Dtype counts:")
print(df.dtypes.value_counts())
print()


pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

df.head()

Shape: (199523, 42)

Dtype counts:
str        29
int64      12
float64     1
Name: count, dtype: int64



,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,race,hispanic origin,sex,member of a labor union,reason for unemployment,full or part time employment stat,capital gains,capital losses,dividends from stocks,tax filer stat,region of previous residence,state of previous residence,detailed household and family stat,detailed household summary in household,weight,migration code-change in msa,migration code-change in reg,migration code-move within reg,live in this house 1 year ago,migration prev res in sunbelt,num persons worked for employer,family members under 18,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Other Rel 18+ ever marr not in subfamily,Other relative of householder,1700.09,?,?,?,Not in universe under 1 year old,?,0,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,White,All other,Male,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Head of household,South,Arkansas,Householder,Householder,1053.55,MSA to MSA,Same county,Same county,No,Yes,1,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,Asian or Pacific Islander,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Child 18+ never marr Not in a subfamily,Child 18 or older,991.95,?,?,?,Not in universe under 1 year old,?,0,Not in universe,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1758.14,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1069.16,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.


## Initial Observations

| # | Observation | Next step |
|---|---|---|
| 1 | 199,523 rows × 42 columns. Layout matches the brief: 40 features + `weight` + `label`. | Confirm the role of each column by name. |
| 2 | Type mix: 29 string, 12 int, 1 float (`weight`). Several int columns look like categorical recodes by name. | Identify which int columns are true categorical recodes (e.g., `detailed industry recode`, `detailed occupation recode`, `year`). |
| 3 | Two distinct sentinel forms appear in `head()`: `?` and `Not in universe` (with variants like `Not in universe or children`). | Per-column sentinel scan: count `?`, `Not in universe`, and any other non-canonical values. |
| 4 | Label is a string (e.g., `- 50000.`); `head()` rows show only one class. | Compute target distribution (raw + weighted); recode to a clean binary indicator. |
| 5 | `weight` column is a float, consistent with stratified sampling per the brief. | Quantify weight distribution and decide whether to use it in training, evaluation, or both. |

## Column taxonomy

- Address initial observation #1 & #2.
- Classify each of the 42 columns by **role** (feature / target / weight) and **modeling type** (categorical / numeric). Several integer-typed columns are categorical recodes by name, flag them now so they get treated correctly downstream.

In [6]:
# Build a starter taxonomy: role + base modeling type for every column
def initial_role(name: str) -> str:
    if name == "label":
        return "target"
    if name == "weight":
        return "weight"
    return "feature"

def base_modeling_type(dtype) -> str:
    return "numeric" if pd.api.types.is_numeric_dtype(dtype) else "categorical"

taxonomy = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "nunique": [df[c].nunique() for c in df.columns],
    "role": [initial_role(c) for c in df.columns],
    "modeling_type": [base_modeling_type(df[c].dtype) for c in df.columns],
})

taxonomy

,column,dtype,nunique,role,modeling_type
0,age,int64,91,feature,numeric
1,class of worker,str,9,feature,categorical
2,detailed industry recode,int64,52,feature,numeric
3,detailed occupation recode,int64,47,feature,numeric
4,education,str,17,feature,categorical
5,wage per hour,int64,1240,feature,numeric
6,enroll in edu inst last wk,str,3,feature,categorical
7,marital stat,str,7,feature,categorical
8,major industry code,str,24,feature,categorical
9,major occupation code,str,15,feature,categorical


In [7]:
# Inspect int columns by cardinality
# Low-cardinality int columns are likely categorical recodes despite the int dtype
int_cols = [c for c in df.columns if pd.api.types.is_integer_dtype(df[c])]
int_summary = pd.DataFrame({
    "column": int_cols,
    "nunique": [df[c].nunique() for c in int_cols],
    "sample_values": [sorted(df[c].unique())[:10] for c in int_cols],
})
int_summary.sort_values("nunique").reset_index(drop=True)

,column,nunique,sample_values
0,year,2,"[94, 95]"
1,own business or self employed,3,"[0, 1, 2]"
2,veterans benefits,3,"[0, 1, 2]"
3,num persons worked for employer,7,"[0, 1, 2, 3, 4, 5, 6]"
4,detailed occupation recode,47,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
5,detailed industry recode,52,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
6,weeks worked in year,53,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
7,age,91,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]"
8,capital losses,113,"[0, 155, 213, 323, 419, 625, 653, 772, 810, 880]"
9,capital gains,132,"[0, 114, 401, 594, 914, 991, 1055, 1086, 1090, 1111]"


In [16]:
# Override int columns to be categorical recodes
CATEGORICAL_INT_OVERRIDES = [
    "year",
    "own business or self employed",
    "veterans benefits",
    "detailed industry recode",
    "detailed occupation recode", 
]

taxonomy.loc[
    taxonomy["column"].isin(CATEGORICAL_INT_OVERRIDES),
    "modeling_type",
] = "categorical"


taxonomy_counts = (
    taxonomy
    .groupby(["role", "modeling_type"])
    .size()
    .reset_index(name="count")
)

print("Taxonomy counts:")
print(taxonomy_counts.to_string(index=False))
print()

taxonomy

Taxonomy counts:
   role modeling_type  count
feature   categorical     33
feature       numeric      7
 target   categorical      1
 weight       numeric      1



,column,dtype,nunique,role,modeling_type
0,age,int64,91,feature,numeric
1,class of worker,str,9,feature,categorical
2,detailed industry recode,int64,52,feature,categorical
3,detailed occupation recode,int64,47,feature,categorical
4,education,str,17,feature,categorical
5,wage per hour,int64,1240,feature,numeric
6,enroll in edu inst last wk,str,3,feature,categorical
7,marital stat,str,7,feature,categorical
8,major industry code,str,24,feature,categorical
9,major occupation code,str,15,feature,categorical


### Observations

| # | Observation | Next step |
|---|---|---|
| 1 | 42 columns classified: 1 target (`label`), 1 weight (`weight`), 40 features. | Use `taxonomy` as the reference for downstream preprocessing. |
| 2 | 28 string features are unambiguously categorical. | Carry to sentinel scan. |
| 3 | 5 int features are confirmed categorical recodes by low cardinality and code-like values: `detailed industry recode` (52), `detailed occupation recode` (47), `year` (2), `own business or self employed` (3), `veterans benefits` (3). | Override applied already. |
| 4 | `num persons worked for employer` has 7 values (0–6) — borderline. The naming reads as a count, so it stays numeric. | Revisit if model behavior on it looks off. |
| 5 | Final taxonomy: **33 categorical features + 7 numeric features**, plus 1 target and 1 weight. | Proceed to per-column sentinel/missingness scan (categorical). |
| 6 | 7 numeric features identified — distributional shape, outliers, and zero-spike behavior need inspection before any scaling or transformation decision. | Per-column `describe()` + histograms; flag zero-spikes and heavy tails. |

## Categorical column scan

- Address initial observation #3.
- For each of the 33 categorical features, count sentinel forms (`?`, `Not in universe*`, `NaN`, `Do not know`) and surface top values. Decide per-column treatment for sentinels.

In [ ]:
# Sentinel summary across all categorical features
cat_features = taxonomy.loc[
    (taxonomy["role"] == "feature") & (taxonomy["modeling_type"] == "categorical"),
    "column",
].tolist()

def sentinel_breakdown(s: pd.Series) -> dict:
    n = len(s)
    nan_n = s.isna().sum()
    s_str = s.dropna().astype(str).str.strip()
    q = (s_str == "?").sum()
    niv = s_str.str.startswith("Not in universe").sum()
    dnk = (s_str == "Do not know").sum()
    return {
        "pct_q": q / n * 100,
        "pct_niv": niv / n * 100,
        "pct_other": (nan_n + dnk) / n * 100,
        "pct_total_sentinel": (q + niv + nan_n + dnk) / n * 100,
    }

rows = []
for c in cat_features:
    s = df[c]
    top = s.value_counts(dropna=False).head(1)
    rows.append({
        "column": c,
        "nunique": s.nunique(),
        "top_value": str(top.index[0])[:35],
        **sentinel_breakdown(s),
    })

cat_summary = (
    pd.DataFrame(rows)
    .sort_values("pct_total_sentinel", ascending=False)
    .reset_index(drop=True)
)
cat_summary.round(2)

,column,nunique,top_value,pct_q,pct_niv,pct_other,pct_total_sentinel
0,fill inc questionnaire for veteran's admin,3,Not in universe,0.00,99.01,0.00,99.01
1,reason for unemployment,6,Not in universe,0.00,96.96,0.00,96.96
2,enroll in edu inst last wk,3,Not in universe,0.00,93.69,0.00,93.69
3,state of previous residence,51,Not in universe,0.35,92.09,0.00,92.45
4,region of previous residence,6,Not in universe,0.00,92.09,0.00,92.09
5,migration prev res in sunbelt,4,?,49.97,42.13,0.00,92.09
6,member of a labor union,3,Not in universe,0.00,90.45,0.00,90.45
7,family members under 18,5,Not in universe,0.00,72.29,0.00,72.29
8,live in this house 1 year ago,3,Not in universe under 1 year old,0.00,50.73,0.00,50.73
9,migration code-move within reg,10,?,49.97,0.76,0.00,50.73


In [ ]:
# Spot-check value distribution on the top-5 sentinel-heaviest columns.
for c in cat_summary.head(5)["column"]:
    print(f"--- {c} (nunique={df[c].nunique(dropna=False)}) ---")
    print(df[c].value_counts(dropna=False).head(8))
    print()

### Observations

| # | Observation | Next step |
|---|---|---|
| 1 | 14 of 33 categorical features have **zero sentinel content**; the remaining 19 carry sentinels in some form. | The 14 clean columns can be one-hot encoded directly. |
| 2 | 11 features have **≥50% `Not in universe*`** — these are conditional questions where the sentinel itself signals "doesn't apply" (e.g., `reason for unemployment` only applies if unemployed; `enroll in edu inst last wk` only to students). | **Keep `Not in universe*` as its own category** in encoding — it carries information, do not treat as missing. |
| 3 | Three migration-code columns (`migration code-change in msa`, `... in reg`, `... move within reg`) and `migration prev res in sunbelt` show ~49.97% `?` — a structural pattern (about half the population has no migration record). | Treat `?` as a separate "missing" category (do not impute, do not drop). |
| 4 | `country of birth father/mother/self` show 1.70–3.36% `?` — scattered true missingness, not structural. | Treat `?` as a separate "unknown" category. (Imputation from within-family is possible but biased; flagging is safer.) |
| 5 | `hispanic origin` is the only column with both `NaN` (874) and `"Do not know"` (306) — combined ≈0.59%. | Recode `NaN` and `"Do not know"` into a single `Unknown` category. |
| 6 | **Drop / consolidation candidates** flagged for empirical testing (not dropped now). <br><br>**Redundancy pairs** (detailed vs. coarse — same signal, different granularity): `detailed industry recode` (52) vs `major industry code` (24); `detailed occupation recode` (47) vs `major occupation code` (15); `detailed household and family stat` (38) vs `detailed household summary in household` (8); `state of previous residence` (51) vs `region of previous residence` (6). <br><br>**Likely correlated**: 3 migration-code columns (`change in msa`, `change in reg`, `move within reg`); 3 country-of-birth columns (`father`, `mother`, `self`). <br><br>**Likely uninformative**: `year` (only 94/95 — survey artifact, not predictive signal). <br><br>**High cardinality + heavy sentinel** (one-hot expensive for linear models): `state of previous residence` (51 values, 92% Not in universe). | Test each candidate in the modeling notebook via feature importance + ablation. Pre-dropping replaces evidence with intuition; tree-based models in particular absorb redundancy gracefully. |

## Numeric descriptive analysis

- Address Column Taxonomy observation #6.
- For each of the 7 numeric features, examine `describe()` summary statistics, zero-spike pattern, and distribution shape (linear and non-zero) to drive scaling and transformation decisions.

In [ ]:
# Numeric describe() + zero-pct per feature.
numeric_features = taxonomy.loc[
    (taxonomy["role"] == "feature") & (taxonomy["modeling_type"] == "numeric"),
    "column",
].tolist()

desc = df[numeric_features].describe().T
desc["pct_zero"] = [(df[c] == 0).mean() * 100 for c in numeric_features]
desc.round(2)

In [ ]:
# Histograms — linear-x with log-y so both the zero-spike and the tail are visible.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, c in enumerate(numeric_features):
    axes[i].hist(df[c], bins=30, color="steelblue", edgecolor="white")
    axes[i].set_yscale("log")
    axes[i].set_title(c, fontsize=10)

axes[7].axis("off")
fig.suptitle("Numeric features — linear-x, log-y histograms (30 bins)", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# Non-zero distribution for the 4 heavy-tailed columns — the actual signal lives here.
heavy_tailed = ["wage per hour", "capital gains", "capital losses", "dividends from stocks"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, c in enumerate(heavy_tailed):
    nz = df[df[c] > 0][c]
    axes[i].hist(nz, bins=50, color="steelblue", edgecolor="white")
    axes[i].set_yscale("log")
    axes[i].set_title(f"{c}\n(non-zero, n={len(nz):,})", fontsize=10)

fig.suptitle("Heavy-tailed numerics — non-zero subset (linear-x, log-y, 50 bins)", fontsize=12)
fig.tight_layout()
plt.show()

### Observations

| # | Observation | Next step |
|---|---|---|
| 1 | `age`, `num persons worked for employer`, `weeks worked in year` are well-behaved (bounded ranges, count semantics). | Pass through unscaled to tree models; standardize for linear models. |
| 2 | Four features are heavily zero-inflated: `capital losses` (98.04% zero), `capital gains` (96.30%), `wage per hour` (94.33%), `dividends from stocks` (89.40%). The non-zero minority is where the signal lives. | Consider engineering a `has_X > 0` binary flag alongside the raw value, or a log-transform with offset. Tree models can absorb the spike directly; linear models need transformation. |
| 3 | `capital gains` shows **clear top-coding**: 390 records (5.29% of non-zero) sit at the max value `99999`. `dividends from stocks` has 25 records at `99999` (marginal top-coding). `wage per hour` (1 record at 9999) and `capital losses` (4 at 4608) appear to have legitimate maxima. | For `capital gains`: treat `99999` as a separate "≥cap" indicator, or cap at a lower value (e.g., 99th percentile of non-zero). Decide empirically. |
| 4 | `num persons worked for employer` and `weeks worked in year` share 48.11% zeros — the "not in the labor force" population. This matches the categorical sentinel rate for `class of worker` (50.24% `Not in universe`), pointing to consistent encoding across types. | The numeric-zero here is semantically equivalent to `Not in universe` elsewhere. A derived "in labor force" indicator could consolidate the signal across columns; test in modeling. |
| 5 | Non-zero distributions for `capital gains` (mean ≈11,755 among 3.70% with gains) and `dividends from stocks` (mean ≈1,864 among 10.60% with dividends) are heavily right-skewed. | Log-transform these for linear-model use. |